In [1]:
import Pkg

function find_flexopt_root(start_dir=joinpath(homedir(), "flexOPT"))
    candidates = String[]

    if haskey(ENV, "FLEXOPT_ROOT")
        push!(candidates, abspath(expanduser(ENV["FLEXOPT_ROOT"])))
    end

    directory = abspath(expanduser(start_dir))

    while true
        push!(candidates, directory)
        parent = dirname(directory)
        parent == directory && break
        directory = parent
    end

    for candidate in unique(candidates)
        project_file = joinpath(candidate, "Project.toml")
        source_file = joinpath(candidate, "src", "commonBatchs.jl")

        if isfile(project_file) && isfile(source_file)
            return candidate
        end
    end

    error("Cannot locate flexOPT. Set ENV[\"FLEXOPT_ROOT\"] to its absolute path.")
end

flexopt_root = find_flexopt_root()
Pkg.activate(flexopt_root)

@show VERSION Threads.nthreads() Base.active_project()

  Activating 

VERSION = v"1.12.6"
Threads.nthreads() = 1
Base.active_project() = "/home/ydenizhernandez/flexOPT/Project.toml"


project at `~/flexOPT`


"/home/ydenizhernandez/flexOPT/Project.toml"

In [2]:
Pkg.add("IJulia")
Pkg.add("PythonCall")
ENV["JULIA_PYTHONCALL_EXE"] = "/usr/bin/python3" #Where should the user look for this?
ENV["JULIA_CONDAPKG_BACKEND"] = "Null"

using PythonCall

pybuiltins = pyimport("builtins")  # Access Python's built-in functions (e.g., str(), type()) to handle Python objects returned by cppyy.
pyimport("sys").executable

   Resolving package versions...
     Project No packages added to or removed from `~/flexOPT/Project.toml`
    Manifest No packages added to or removed from `~/flexOPT/Manifest.toml`
Precompiling packages...
   8463.4 ms  ✓ InlineStrings → ParsersExt
    676.0 ms  ✓ WeakRefStrings
  12066.7 ms  ✓ Conda
  12187.6 ms  ✓ ColorBrewer
  16535.7 ms  ✓ BenchmarkTools
  13197.2 ms  ✓ CSV
  35792.6 ms  ✓ SciMLBase → SciMLBasePythonCallExt
  56096.0 ms  ✓ Plots
   2464.3 ms  ✓ Plots → IJuliaExt
   2559.9 ms  ✓ Plots → FileIOExt
   2719.5 ms  ✓ Plots → GeometryBasicsExt
   2820.9 ms  ✓ Plots → UnitfulExt
   3564.9 ms  ✓ StatGeochem → StatGeochemPlotsExt
 102042.1 ms  ✓ Makie
   4808.3 ms  ✓ DomainSets → DomainSetsMakieExt
   5076.8 ms  ✓ SciMLBase → SciMLBaseMakieExt
  32512.9 ms  ✓ CairoMakie
  62753.8 ms  ✓ GLMakie
  18 dependencies successfully precompiled in 180 seconds. 687 already precompiled.
   Resolving package versions...
     Project No packages added to or removed from `~/flexOPT/Pro

Python: '/usr/bin/python3'

In [3]:
os = pyimport("os")
sys = pyimport("sys")

ROOT_LIB = "/home/ydenizhernandez/root-install/lib" #Should be the path to the root library, not the root source code. This is where the ROOT.so file is located.
PROJECT_ROOT = "/home/ydenizhernandez/ModularEarth" # Shoukd be the path to the root source code, where the CMakeLists.txt file is located.

# Match the configuration of your working Python notebook
sys.path.insert(0, ROOT_LIB)
os.chdir(PROJECT_ROOT)

# Attempt to import CERN ROOT
ROOT = pyimport("ROOT")

println("ROOT version: ", ROOT.gROOT.GetVersion())


ROOT version: 6.36.000


In [4]:
# ── Step 1: Load the compiled ModularEarth shared library ──────────────────
load_status = ROOT.gSystem.Load("./libModularEarth.so")
# 0 = loaded successfully, 1 = already loaded, negative = error
println("Library load status : ", pyconvert(Int, load_status),
        "  (0 = success, 1 = already loaded)")

ME = ROOT.ModularEarth
println("ModularEarth namespace bound : ", ME)

Library load status : 0  (0 = success, 1 = already loaded)
ModularEarth namespace bound : <namespace cppyy.gbl.ModularEarth at 0x4af77020>


In [5]:
# ── Step 2: Construct Earth and load a PREM table ───────────────────────────
# prem_15layers is the smallest available table — builds quickly
earth = ME.Earth("JuliaTest", "prem_test", 60.0, 60.0)
println("Earth object created")

# Populate layersconfigs from the PREM table
earth.SetEarthModel("prem_test")

println("\n--- Model Summary ---")
earth.GetModelSummary()

Earth object created


Info in <TGeoManager::TGeoManager>: Geometry JuliaTest, JuliaTest Earth Model created
Info in <TGeoManager::SetTopVolume>: Top volume is EarthTopSphere. Master volume is EarthTopSphere
Info in <TGeoNavigator::BuildCache>: --- Maximum geometry depth set to 100



--- Model Summary ---

--- Layer 1 (i_ly = 1) ---
  Sublayer 1 | rmin: 0 | rmax: 1221.5 | ρ: 12.9807 | Z/A: 0.4691

--- Layer 2 (i_ly = 2) ---
  Sublayer 1 | rmin: 1221.5 | rmax: 3480 | ρ: 10.0861 | Z/A: 0.4691

--- Layer 3 (i_ly = 3) ---
  Sublayer 1 | rmin: 3480 | rmax: 5701 | ρ: 4.4872 | Z/A: 0.4954

--- Layer 4 (i_ly = 4) ---
  Sublayer 1 | rmin: 5701 | rmax: 5971 | ρ: 3.8983 | Z/A: 0.4954

--- Layer 5 (i_ly = 5) ---
  Sublayer 1 | rmin: 5971 | rmax: 6356 | ρ: 2.9 | Z/A: 0.4956

--- Layer 6 (i_ly = 6) ---
  Sublayer 1 | rmin: 6356 | rmax: 6371 | ρ: 1.02 | Z/A: 0.5525

--- Layer 7 (i_ly = 7) ---
  Sublayer 1 | rmin: 6371 | rmax: 6386 | ρ: 0.001 | Z/A: 0.4991


Python: None

## `SetSublayerUpdates` — Customising a sublayer before building

`SetSublayerUpdates` lets you overwrite the material properties of any individual sublayer in the model **after** loading a PREM table with `SetEarthModel` but **before** calling `BuiltEarth()`.  The change is applied directly to the internal `layersconfigs` vector, so the modified values are what ROOT sees when it constructs the geometry.

### Signature
```cpp
earth.SetSublayerUpdates(
    i_ly,               # Int  — 1-based index of the major geological layer
    i_sly,              # Int  — 1-based index of the sublayer within that layer
    rho_sly,            # Float64 — new bulk density  [g/cm³]
    eyield_sly,         # Float64 — new electron yield  Yₑ = Σ wᵢ·(Z/A)ᵢ
    UpdateMaterialName  # String  — label for the new ROOT TGeoMaterial
)
```

### Usage notes
| Rule | Detail |
|------|--------|
| **Must precede `BuiltEarth()`** | Calling it after the geometry is already built raises a warning and does nothing. |
| **Indices are 1-based** | Major layer 1 = Inner Core; check `GetModelSummary()` for the full index list. |
| **`atomicMass` is fixed to 1.0** | The implementation encodes the electron yield directly in the `atomicNumber` field and sets `atomicMass = 1`, matching the convention used by the rest of the library. |
| **Verify with `GetModelSummary()`** | Call it before *and* after to confirm the mutation was applied. |

### Quick example
```julia
# Override Inner Core (layer 0, sublayer 0) density and electron yield
earth.SetSublayerUpdates(1, 1, 10.0, 0.45, "MyCustomMaterial")
earth.GetModelSummary()   # confirm the change
earth.BuiltEarth()        # build geometry with updated values
```

In [6]:
# ── Test: SetSublayerUpdates — pre-build config mutation check ─────────────
# SetSublayerUpdates must be called BEFORE BuiltEarth().
# GetModelSummary() reads directly from layersconfigs, so comparing its output
# before and after the call is the ground-truth check that the config was mutated.
#
# Target  : Layer 1 (Inner Core), Sublayer 1
# Original: ρ = 12.9807 g/cm³,  Z/A = 0.4691
# Updated : ρ = 10.0   g/cm³,  Z/A = 0.45

test_ly   = 1       # Inner Core — major layer index (1-based)
test_sly  = 1       # first sublayer of the Inner Core (1-based)
orig_rho  = 12.9807
orig_ye   = 0.4691
new_rho   = 10.0
new_ye    = 0.45
new_name  = "TestMaterial_InnerCore"

println("─── BEFORE SetSublayerUpdates ─────────────────────────────────────────")
earth.GetModelSummary()

# earth.SetSublayerUpdates(test_ly, test_sly, new_rho, new_ye, new_name)

println("\n─── AFTER SetSublayerUpdates ──────────────────────────────────────────")
earth.GetModelSummary()
println("\nExpected Layer $test_ly Sublayer $test_sly: ρ = $new_rho  Z/A = $new_ye  name = $new_name")

# ── Expected output (Layer 0 Sublayer 1 line changes; all others stay the same)
# ─── BEFORE SetSublayerUpdates ─────────────────────────────────────────
# --- Layer index: 0Index check: 1 ---
#   Sublayer 1 | rmin: 0 | rmax: 1221.5 | ρ: 12.9807 | Z/A: 0.4691
# ...
#
# ─── AFTER SetSublayerUpdates ──────────────────────────────────────────
# --- Layer index: 0Index check: 1 ---
#   Sublayer 1 | rmin: 0 | rmax: 1221.5 | ρ: 10.0 | Z/A: 0.45
# ...
# Expected Layer 0 Sublayer 0: ρ = 10.0  Z/A = 0.45  name = TestMaterial_InnerCore

─── BEFORE SetSublayerUpdates ─────────────────────────────────────────

─── AFTER SetSublayerUpdates ──────────────────────────────────────────

Expected Layer 1 Sublayer 1: ρ = 10.0  Z/A = 0.45  name = TestMaterial_InnerCore

--- Layer 1 (i_ly = 1) ---
  Sublayer 1 | rmin: 0 | rmax: 1221.5 | ρ: 12.9807 | Z/A: 0.4691

--- Layer 2 (i_ly = 2) ---
  Sublayer 1 | rmin: 1221.5 | rmax: 3480 | ρ: 10.0861 | Z/A: 0.4691

--- Layer 3 (i_ly = 3) ---
  Sublayer 1 | rmin: 3480 | rmax: 5701 | ρ: 4.4872 | Z/A: 0.4954

--- Layer 4 (i_ly = 4) ---
  Sublayer 1 | rmin: 5701 | rmax: 5971 | ρ: 3.8983 | Z/A: 0.4954

--- Layer 5 (i_ly = 5) ---
  Sublayer 1 | rmin: 5971 | rmax: 6356 | ρ: 2.9 | Z/A: 0.4956

--- Layer 6 (i_ly = 6) ---
  Sublayer 1 | rmin: 6356 | rmax: 6371 | ρ: 1.02 | Z/A: 0.5525

--- Layer 7 (i_ly = 7) ---
  Sublayer 1 | rmin: 6371 | rmax: 6386 | ρ: 0.001 | Z/A: 0.4991

--- Layer 1 (i_ly = 1) ---
  Sublayer 1 | rmin: 0 | rmax: 1221.5 | ρ: 12.9807 | Z/A: 0.4691

--- Layer 2 (i_ly = 2) ---
  Su

In [7]:
# ── Step 3: Build the geometry tree and close it ────────────────────────────
# BuiltEarth() constructs Layer/Sublayer volumes, adds them to the top
# volume, then calls CloseGeometry() through the compiled C++ wrapper.
earth.BuiltEarth()
println("BuiltEarth() returned")

TGeoMaterial::ctor:0: RuntimeWarning: Material BlockMat_1_1_1 defined with fractional Z=0.469100
TGeoMaterial::ctor:0: RuntimeWarning: Material BlockMat_2_1_1 defined with fractional Z=0.469100
TGeoMaterial::ctor:0: RuntimeWarning: Material BlockMat_2_1_2 defined with fractional Z=0.469100
TGeoMaterial::ctor:0: RuntimeWarning: Material BlockMat_2_1_3 defined with fractional Z=0.469100
TGeoMaterial::ctor:0: RuntimeWarning: Material BlockMat_2_1_4 defined with fractional Z=0.469100
TGeoMaterial::ctor:0: RuntimeWarning: Material BlockMat_2_1_5 defined with fractional Z=0.469100
TGeoMaterial::ctor:0: RuntimeWarning: Material BlockMat_2_1_6 defined with fractional Z=0.469100
TGeoMaterial::ctor:0: RuntimeWarning: Material BlockMat_2_1_7 defined with fractional Z=0.469100
TGeoMaterial::ctor:0: RuntimeWarning: Material BlockMat_2_1_8 defined with fractional Z=0.469100
TGeoMaterial::ctor:0: RuntimeWarning: Material BlockMat_2_1_9 defined with fractional Z=0.469100
TGeoMaterial::ctor:0: RuntimeW

BuiltEarth() returned


In [8]:
# ── Step 4: Verify geometry finalization ────────────────────────────────────
geo = earth.GetGeoManager()
top = earth.GetTopVolume()

geo_name = pyconvert(String, geo.GetName())
top_name = pyconvert(String, top.GetName())
is_closed = pyconvert(Bool, geo.IsClosed())
n_daughters = pyconvert(Int, top.GetNdaughters())

println("TGeoManager name   : ", geo_name)
println("Top volume name    : ", top_name)
println("IsClosed           : ", is_closed)
println("Top daughters      : ", n_daughters, " (one per major layer)")

# Cross-check: ROOT keeps a global gGeoManager; it should be Earth's manager
global_geo = ROOT.gGeoManager
global_name = pyconvert(String, global_geo.GetName())
println("\nglobal gGeoManager : ", global_name)
println("Same as Earth's?   : ", geo_name == global_name)

@assert is_closed "FAIL: geometry was not closed"
@assert n_daughters > 0 "FAIL: top volume has no daughters"
println("\nAll assertions passed.")

TGeoManager name   : JuliaTest
Top volume name    : EarthTopSphere
IsClosed           : true
Top daughters      : 7 (one per major layer)

global gGeoManager : JuliaTest
Same as Earth's?   : true

All assertions passed.


In [9]:
# ── Test: SetSublayerUpdates — post-build ROOT geometry verification ────────
# BuiltEarth() constructed the ROOT geometry from the mutated layersconfigs.
# Navigate to (0, 0, 0) — centre of the Inner Core — and confirm that the
# material density and name in the live ROOT geometry match the values written
# by SetSublayerUpdates (new_rho = 10.0 g/cm³, new_name = "TestMaterial_InnerCore").

geo = earth.GetGeoManager()

node_at_origin = geo.FindNode(0.0, 0.0, 0.0)
mat            = geo.GetCurrentVolume().GetMedium().GetMaterial()

mat_name    = pyconvert(String,  mat.GetName())
mat_density = pyconvert(Float64, mat.GetDensity())

println("Node at (0,0,0)  : ", pyconvert(String, node_at_origin.GetName()))
println("Material name    : ", mat_name)
println("Material density : ", mat_density, " g/cm³")
println("Expected density : ", new_rho,     " g/cm³")
println("Expected name    : ", new_name)


# ── Expected output ──────────────────────────────────────────────────────────
# Node at (0,0,0)  : BlockVolume_1_1
# Material name    : TestMaterial_InnerCore
# Material density : 10.0 g/cm³
# Expected density : 10.0 g/cm³
# Expected name    : TestMaterial_InnerCore
#
# Both assertions passed — SetSublayerUpdates propagated correctly to ROOT geometry.

Node at (0,0,0)  : BlockVol_1_1_1_1
Material name    : BlockMat_1_1_1
Material density : 12.9807 g/cm³
Expected density : 10.0 g/cm³
Expected name    : TestMaterial_InnerCore


In [10]:
# ── Step 5: Smoke-test geometry navigation ──────────────────────────────────
# Navigate to the master volume; this fails if the geometry is not closed.
geo.CdTop()
current_vol = pyconvert(String, geo.GetCurrentVolume().GetName())
println("Current volume after cd(\"/\") : ", current_vol)

# FindNode is a basic navigation sanity check
found = geo.FindNode(0.0, 0.0, 0.0)   # centre of the Earth
node_name = pyconvert(String, found.GetName())
println("Node at (0,0,0)              : ", node_name)

println("\nGeometry is closed and navigable from Julia via PythonCall/PyROOT.")

Current volume after cd("/") : EarthTopSphere
Node at (0,0,0)              : BlockVol_1_1_1_1

Geometry is closed and navigable from Julia via PythonCall/PyROOT.


---
## `EarthPaths` — Neutrino Geometric Path Through the Earth

`EarthPaths` computes the sequence of ROOT geometry segments that a neutrino traverses
between an atmospheric entry point and a surface detector.  It reuses the
`TGeoManager` already built by `Earth::BuiltEarth()` — no second geometry is created.

| Method | Purpose |
|--------|---------|
| `EarthPaths(geoManager)` | Constructor — binds to an existing closed geometry |
| `SetDetectorLocation(lon, lat, R_det)` | Geographic position of the detector |
| `GetSegments(th, phi[, radians])` | Run the tracking loop; fill segment list |
| `PrintSegments()` | Print formatted table to stdout |
| `GetPathSegments()` | Access the `std::vector<PathSegment>` directly |
| `DisplayVectors()` | Draw geometry + path + markers on a ROOT canvas |

### Test 1 — Constructor

`EarthPaths` takes a raw `TGeoManager*` — obtained here via `Earth::GetGeoManager()`.
After construction the object is valid but holds no segments; `GetPathSegments()` returns
an empty vector.

In [11]:
# ── EarthPaths Test 1: Constructor ──────────────────────────────────────────
geo = earth.GetGeoManager()
paths = ME.EarthPaths(geo)
println("EarthPaths object : ", paths)

# Before GetSegments() the segment list must be empty
segs_empty = paths.GetPathSegments()
n_before = pyconvert(Int, segs_empty.size())
println("Segments before GetSegments() : ", n_before)
@assert n_before == 0 "FAIL: expected 0 segments before GetSegments()"
println("\nTest 1 passed — EarthPaths constructed; segment list is empty.")

EarthPaths object : <cppyy.gbl.ModularEarth.EarthPaths object at 0x4af13c60>
Segments before GetSegments() : 0

Test 1 passed — EarthPaths constructed; segment list is empty.


### Test 2 — `SetDetectorLocation`

Sets the detector position from geographic coordinates and builds the local
East / North / Up coordinate frame used by `GetSegments`.

```julia
paths.SetDetectorLocation(
    lon,    # longitude [degrees], East positive
    lat,    # latitude  [degrees], North positive
    R_det   # distance from Earth centre [km]
)
```

We place the detector at the **geographic North Pole** (lat = 90°) on the
Earth's surface (R = 6371 km).  This simplifies verification: the local
+z axis (radially outward) points along the global +Z axis, making the
expected Cartesian coordinates trivially `(0, 0, 6371)`.

In [12]:
# ── EarthPaths Test 2: SetDetectorLocation ───────────────────────────────────
det_lon = 0.0    # longitude  [deg]
det_lat = 90.0   # latitude   [deg]  — North Pole
det_R   = 6371.0 # radius     [km]   — Earth's mean surface radius

paths.SetDetectorLocation(det_lon, det_lat, det_R)
println("SetDetectorLocation called: lon=", det_lon, "° lat=", det_lat, "° R=", det_R, " km")

# Verify by checking that calling GetSegments later does not raise
# (a full assertion on r_det is done in Test 3 via path length)
println("\nTest 2 passed — SetDetectorLocation completed without error.")

SetDetectorLocation called: lon=0.0° lat=90.0° R=6371.0 km

Test 2 passed — SetDetectorLocation completed without error.


### Test 3 — `GetSegments` (degrees, straight up-going)

With the detector at the North Pole, a zenith angle **θ = 180°** points
the neutrino direction straight upward (+Z).  The neutrino source is placed
at the intersection of the atmospheric sphere (R_ATM = 6386 km) with the
−Z axis, i.e. `(0, 0, −6386)`.  The path crosses the entire Earth diameter.

Expected sanity checks:
- `N_segments ≥ 14`  (at least one block per sublayer in prem\_15layers)
- Every segment: `distance > 0`, `i_ly ≥ 1`, `i_sly ≥ 1`, `i_blk ≥ 1`
- Step numbers run **1, 2, 3, …** without gaps
- Total path length **≈ 12 757 km** (= R_ATM + R_det, to within 5 %)

In [13]:
# ── EarthPaths Test 3: GetSegments in degrees ────────────────────────────────
th_deg  = 180.0   # zenith 180° → straight up-going (full Earth diameter)
phi_deg = 0.0



paths.GetSegments(th_deg, phi_deg)   # radians=false is the default

segs = paths.GetPathSegments()



Python: vector<ModularEarth::PathSegment>{ @0x4aa99be0, @0x4aa99c68, @0x4aa99cf0, @0x4aa99d78, @0x4aa99e00, @0x4aa99e88, @0x4aa99f10, @0x4aa99f98, @0x4aa9a020, @0x4aa9a0a8, @0x4aa9a130, @0x4aa9a1b8 }

In [14]:

# ── EarthPaths Test 3: GetSegments in degrees ────────────────────────────────
th_deg  = 180.0   # zenith 180° → straight up-going (full Earth diameter)
phi_deg = 0.0

paths.GetSegments(th_deg, phi_deg)   # radians=false is the default

segs = paths.GetPathSegments()
N    = pyconvert(Int, segs.size())
println("Number of segments recorded : ", N)

# Validate every segment field
prev_step = 0
prev_sum  = 0.0
for i in 0:(N-1)
    s = segs[i]
    step     = pyconvert(Int,    s.step)
    dist     = pyconvert(Float64, s.distance)
    sum      = pyconvert(Float64, s.cumulative)
    i_blk    = pyconvert(Int,    s.i_blk)
    i_sly    = pyconvert(Int,    s.i_sly)
    i_ly     = pyconvert(Int,    s.i_ly)
    vol_name = pyconvert(String, pybuiltins.str(s.volName))
    mat_name = pyconvert(String, pybuiltins.str(s.matName))
    density  = pyconvert(Float64, s.density)
    Z        = pyconvert(Float64, s.Z)
    A        = pyconvert(Float64, s.A)

    @assert step == prev_step + 1  "FAIL: step not sequential at i=$i (got $step, expected $(prev_step+1))"
    @assert dist >  0.0            "FAIL: non-positive distance at step $step ($dist)"
    @assert sum  >  prev_sum       "FAIL: cumulative not increasing at step $step"
    @assert abs(sum - (prev_sum + dist)) < 1e-9 "FAIL: cumulative inconsistent at step $step"
    @assert i_ly  >= 1             "FAIL: invalid layer index at step $step ($i_ly)"
    @assert i_sly >= 1             "FAIL: invalid sublayer index at step $step ($i_sly)"
    @assert i_blk >= 1             "FAIL: invalid block index at step $step ($i_blk)"
    @assert density >= 0.0         "FAIL: negative density at step $step ($density)"
    @assert Z > 0.0                "FAIL: Z <= 0 at step $step ($Z)"
    @assert A > 0.0                "FAIL: A <= 0 at step $step ($A)"
    @assert startswith(vol_name, "BlockVol") "FAIL: unexpected volume name at step $step ($vol_name)"

    prev_step = step
    prev_sum  = sum
    println("Cumulative path length           : ", round(prev_sum; digits=3), " km")
end

# Use the last segment's cumulative as total — avoids floating-point acsumulation drift
total_dist = pyconvert(Float64, segs[N-1].cumulative)
println("Total path length           : ", round(total_dist; digits=3), " km")

# Sanity: total ≈ R_ATM + R_det = 6386 + 6371 = 12 757 km  (within 5 %)
expected_length = 6386.0 + 6371.0
rel_err = abs(total_dist - expected_length) / expected_length
println("Expected (≈)                : ", expected_length, " km")
println("Relative error              : ", round(rel_err * 100; digits=2), " %")
@assert rel_err < 0.05 "FAIL: total path length deviates >5% from expected ($total_dist vs $expected_length)"

println("\nTest 3 passed — ", N, " segments, all fields valid, total length within 5 % of expected.")

Number of segments recorded : 12


LoadError: AssertionError: FAIL: A <= 0 at step 1 (0.0)

### Test — `BlockMaterialCache`: population, single-block setter, sublayer setter

After `BuiltEarth()` the cache is automatically populated from the built geometry.
This test verifies three things in sequence:

1. **Population** — `GetBlockMatInfo(i_ly, i_sly, i_blk).density` matches the PREM table value; `originalDensity == density`; both flags are `false`.
2. **Single-block setter** — `SetBlockDensity(i_ly, i_sly, i_blk, ρ)` updates `density`, sets `isModified = true`, leaves `originalDensity` and `isSubLayerModified` unchanged.
3. **Sublayer setter** — `SetSublayerDensity(i_ly, i_sly, ρ)` updates every block in the sublayer and sets **both** `isModified` and `isSubLayerModified`; blocks in other sublayers are untouched.

In [15]:
# ── BlockMaterialCache Test ───────────────────────────────────────────────────
# prem_test values for reference:
#   Layer 1 (Inner Core) Sublayer 1: ρ = 12.9807 g/cm³  Z/A = 0.4691
#   Layer 2 (Outer Core) Sublayer 1: ρ = 10.0861 g/cm³  Z/A = 0.4691

prem_rho_ic = 12.9807   # expected density for [1,1,1]
prem_ye_ic  = 0.4691    # expected eYield for [1,1,1]

# ── Check 1: Cache populated correctly at BuiltEarth() time ──────────────────
info = earth.GetBlockMatInfo(1, 1, 1)
rho       = pyconvert(Float64, info.density)
orig_rho  = pyconvert(Float64, info.originalDensity)
ye        = pyconvert(Float64, info.eYield)
orig_ye   = pyconvert(Float64, info.originalEYield)
is_mod    = pyconvert(Bool,    info.isModified)
is_slmod  = pyconvert(Bool,    info.isSubLayerModified)

println("─── Check 1: cache population ───────────────────────────────────────────")
println("  density         = $rho  (expected ≈ $prem_rho_ic)")
println("  originalDensity = $orig_rho")
println("  eYield          = $(round(ye; digits=4))  (expected ≈ $prem_ye_ic)")
println("  isModified      = $is_mod  (expected false)")
println("  isSubLayerModified = $is_slmod  (expected false)")

@assert abs(rho - prem_rho_ic)    < 1e-4  "FAIL: cached density ($rho) ≠ PREM ($prem_rho_ic)"
@assert abs(orig_rho - rho)       < 1e-12 "FAIL: originalDensity ($orig_rho) ≠ density at build time ($rho)"
@assert abs(ye - prem_ye_ic)      < 1e-4  "FAIL: cached eYield ($ye) ≠ PREM ($prem_ye_ic)"
@assert abs(orig_ye - ye)         < 1e-12 "FAIL: originalEYield ($orig_ye) ≠ eYield at build time ($ye)"
@assert !is_mod   "FAIL: isModified should be false for unmodified block"
@assert !is_slmod "FAIL: isSubLayerModified should be false for unmodified block"
println("  ✓ passed\n")

# ── Check 2: SetBlockDensity — single block ───────────────────────────────────
new_rho_blk = 99.0
earth.SetBlockDensity(1, 1, 1, new_rho_blk)
info2 = earth.GetBlockMatInfo(1, 1, 1)

rho2      = pyconvert(Float64, info2.density)
orig_rho2 = pyconvert(Float64, info2.originalDensity)
is_mod2   = pyconvert(Bool,    info2.isModified)
is_slmod2 = pyconvert(Bool,    info2.isSubLayerModified)

println("─── Check 2: SetBlockDensity (single block [1,1,1]) ─────────────────────")
println("  density         = $rho2  (expected $new_rho_blk)")
println("  originalDensity = $orig_rho2  (expected $prem_rho_ic — unchanged)")
println("  isModified      = $is_mod2  (expected true)")
println("  isSubLayerModified = $is_slmod2  (expected false — block-level setter)")

@assert abs(rho2 - new_rho_blk)    < 1e-12 "FAIL: density not updated ($rho2 ≠ $new_rho_blk)"
@assert abs(orig_rho2 - prem_rho_ic) < 1e-4 "FAIL: originalDensity was overwritten ($orig_rho2 ≠ $prem_rho_ic)"
@assert is_mod2   "FAIL: isModified should be true after SetBlockDensity"
@assert !is_slmod2 "FAIL: isSubLayerModified should be false after single-block setter"
println("  ✓ passed\n")

# ── Check 3: SetSublayerDensity — entire sublayer ────────────────────────────
target_ly  = 2
target_sly = 1

println("─── Check 3: SetSublayerDensity (sublayer [$target_ly,$target_sly]) ─────")
for blk in [1,2,3,4,5,6,7]
    info_s  = earth.GetBlockMatInfo(target_ly, target_sly, blk)
    rho_s   = pyconvert(Float64, info_s.density)
    mod_s   = pyconvert(Bool,    info_s.isModified)
    slmod_s = pyconvert(Bool,    info_s.isSubLayerModified)
    println("  Block $blk: density=$rho_s  isModified=$mod_s  isSubLayerModified=$slmod_s")
    @assert abs(rho_s - new_rho_sly) < 1e-12 "FAIL: block $blk density not updated by SetSublayerDensity"
    @assert mod_s   "FAIL: block $blk isModified should be true"
    @assert slmod_s "FAIL: block $blk isSubLayerModified should be true"
end


new_rho_sly = 5.0
earth.SetSublayerDensity(target_ly, target_sly, new_rho_sly)

println("─── Check 3: SetSublayerDensity (sublayer [$target_ly,$target_sly]) ─────")
for blk in [1, 2]
    info_s  = earth.GetBlockMatInfo(target_ly, target_sly, blk)
    rho_s   = pyconvert(Float64, info_s.density)
    mod_s   = pyconvert(Bool,    info_s.isModified)
    slmod_s = pyconvert(Bool,    info_s.isSubLayerModified)
    println("  Block $blk: density=$rho_s  isModified=$mod_s  isSubLayerModified=$slmod_s")
    @assert abs(rho_s - new_rho_sly) < 1e-12 "FAIL: block $blk density not updated by SetSublayerDensity"
    @assert mod_s   "FAIL: block $blk isModified should be true"
    @assert slmod_s "FAIL: block $blk isSubLayerModified should be true"
end

# Confirm a block in a different sublayer was NOT touched
info_other  = earth.GetBlockMatInfo(3, 1, 1)
mod_other   = pyconvert(Bool, info_other.isModified)
slmod_other = pyconvert(Bool, info_other.isSubLayerModified)
println("  Layer 3, Sublayer 1, Block 1: isModified=$mod_other  isSubLayerModified=$slmod_other  (expected both false)")
@assert !mod_other   "FAIL: unrelated block was marked isModified"
@assert !slmod_other "FAIL: unrelated block was marked isSubLayerModified"
println("  ✓ passed\n")

println("BlockMaterialCache test passed — population, single-block setter, and sublayer setter all verified.")

─── Check 1: cache population ───────────────────────────────────────────
  density         = 12.9807  (expected ≈ 12.9807)
  originalDensity = 12.9807
  eYield          = 0.4691  (expected ≈ 0.4691)
  isModified      = false  (expected false)
  isSubLayerModified = false  (expected false)
  ✓ passed

─── Check 2: SetBlockDensity (single block [1,1,1]) ─────────────────────
  density         = 99.0  (expected 99.0)
  originalDensity = 12.9807  (expected 12.9807 — unchanged)
  isModified      = true  (expected true)
  isSubLayerModified = false  (expected false — block-level setter)
  ✓ passed

─── Check 3: SetSublayerDensity (sublayer [2,1]) ─────
  Block 1: density=10.0861  isModified=false  isSubLayerModified=false


LoadError: UndefVarError: `new_rho_sly` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

## Block Cache Tests

In [16]:
# TEST 2 - Update are done properly
# ── Cache Structure Check via StatusCacheOriginal ────────────────────────────
# This must run first so we know exact block counts before writing any index-
# specific assertions below.
println("─── StatusCacheOriginal ─────────────────────────────────────────────────")
earth.StatusCacheOriginal()
println("─────────────────────────────────────────────────────────────────────────")
# Visual inspection: confirm 7 layers, 1 sublayer each, block counts > 0.




─── StatusCacheOriginal ─────────────────────────────────────────────────
─────────────────────────────────────────────────────────────────────────
Total Major Layers in model = 7
  Layer 1: Total number of Sublayers: 1
    Sublayer_1_1: Total number of blocks: 1
  Layer 2: Total number of Sublayers: 1
    Sublayer_2_1: Total number of blocks: 18
  Layer 3: Total number of Sublayers: 1
    Sublayer_3_1: Total number of blocks: 18
  Layer 4: Total number of Sublayers: 1
    Sublayer_4_1: Total number of blocks: 18
  Layer 5: Total number of Sublayers: 1
    Sublayer_5_1: Total number of blocks: 18
  Layer 6: Total number of Sublayers: 1
    Sublayer_6_1: Total number of blocks: 18
  Layer 7: Total number of Sublayers: 1
    Sublayer_7_1: Total number of blocks: 18


In [17]:
# ── Test: 5 individual block updates across 5 layers ─────────────────────────
# Target blocks: (ly=1,sly=1,blk=1) through (ly=5,sly=1,blk=1)
# prem_test PREM values used as reference:
#   Layer 1: ρ=12.9807  Ye=0.4691
#   Layer 2: ρ=10.0861  Ye=0.4691
#   Layer 3: ρ=4.4872   Ye=0.4954
#   Layer 4: ρ=3.8983   Ye=0.4954
#   Layer 5: ρ=2.9      Ye=0.4956

targets = [
    (1, 1, 1, 12.9807, 0.4691),   # (i_ly, i_sly, i_blk, orig_rho, orig_ye)
    (2, 1, 1, 10.0861, 0.4691),
    (3, 1, 1,  4.4872, 0.4954),
    (4, 1, 1,  3.8983, 0.4954),
    (5, 1, 1,  2.9000, 0.4956),
]
new_densities = [111.0, 222.0, 333.0, 444.0, 555.0]
new_eyields   = [0.11,  0.22,  0.33,  0.44,  0.55 ]

# Apply updates to BlockMaterialCache
for (idx, (i_ly, i_sly, i_blk, _, _)) in enumerate(targets)
    earth.SetBlockDensity(i_ly, i_sly, i_blk, new_densities[idx])
    earth.SetBlockEYield(i_ly, i_sly, i_blk, new_eyields[idx])
end
println("5 block updates applied.")

5 block updates applied.


In [18]:
# ── Verify: Cache updated, CacheOriginal untouched ───────────────────────────
cache     = earth.GetBlockMaterialCache()
cache_orig = earth.GetBlockMaterialCacheOriginal()

println("─── Block-by-block comparison ───────────────────────────────────────────")
for (idx, (i_ly, i_sly, i_blk, orig_rho, orig_ye)) in enumerate(targets)
    # 0-based indexing into the Python vectors returned through cppyy
    blk_cache = cache[i_ly-1][i_sly-1][i_blk-1]
    blk_orig  = cache_orig[i_ly-1][i_sly-1][i_blk-1]

    rho_c  = pyconvert(Float64, blk_cache.density)
    ye_c   = pyconvert(Float64, blk_cache.eYield)
    rho_o  = pyconvert(Float64, blk_orig.density)
    ye_o   = pyconvert(Float64, blk_orig.eYield)
    mod_c  = pyconvert(Bool,    blk_cache.isModified)

    println("  Block [$i_ly,$i_sly,$i_blk]:")
    println("    Cache     ρ=$rho_c  Ye=$ye_c  isModified=$mod_c")
    println("    Original  ρ=$rho_o  Ye=$ye_o")

    @assert abs(rho_c - new_densities[idx]) < 1e-12 "FAIL: cache density not updated for block [$i_ly,$i_sly,$i_blk]"
    @assert abs(ye_c  - new_eyields[idx])   < 1e-12 "FAIL: cache eYield not updated for block [$i_ly,$i_sly,$i_blk]"
    @assert abs(rho_o - orig_rho) < 1e-4           "FAIL: original density was overwritten for block [$i_ly,$i_sly,$i_blk]"
    @assert abs(ye_o  - orig_ye)  < 1e-4           "FAIL: original eYield was overwritten for block [$i_ly,$i_sly,$i_blk]"
    @assert mod_c "FAIL: isModified should be true for block [$i_ly,$i_sly,$i_blk]"
end
println("\n✓ All 5 blocks: cache updated, original intact.")

─── Block-by-block comparison ───────────────────────────────────────────
  Block [1,1,1]:


    Cache     ρ=111.0  Ye=0.11  isModified=true
    Original  ρ=12.9807  Ye=0.4691
  Block [2,1,1]:
    Cache     ρ=222.0  Ye=0.22  isModified=true
    Original  ρ=10.0861  Ye=0.4691
  Block [3,1,1]:
    Cache     ρ=333.0  Ye=0.33  isModified=true
    Original  ρ=4.4872  Ye=0.4954
  Block [4,1,1]:
    Cache     ρ=444.0  Ye=0.44  isModified=true
    Original  ρ=3.8983  Ye=0.4954
  Block [5,1,1]:
    Cache     ρ=555.0  Ye=0.55  isModified=true
    Original  ρ=2.9  Ye=0.4956

✓ All 5 blocks: cache updated, original intact.


In [19]:
# ── Test: Full sublayer update ────────────────────────────────────────────────
# Target: Layer 6, Sublayer 1 (ocean layer, ρ=1.02, Ye=0.5525 in prem_test)
sly_ly     = 6;  sly_sly   = 1
orig_rho_6 = 1.02;  orig_ye_6 = 0.5525
new_rho_6  = 9.99;  new_ye_6  = 0.99






earth.SetSublayerDensity(sly_ly, sly_sly, new_rho_6)
earth.SetSublayerEYield(sly_ly, sly_sly, new_ye_6)
println("Sublayer [$sly_ly,$sly_sly] update applied.")

# Check every block in the sublayer via the vector getter
cache2    = earth.GetBlockMaterialCache()
cache_o2  = earth.GetBlockMaterialCacheOriginal()
n_blocks  = pyconvert(Int, cache2[sly_ly-1][sly_sly-1].size())

println("─── Sublayer [$sly_ly,$sly_sly]: $n_blocks blocks ────────────────────────────────")
for b in 0:(n_blocks-1)
    blk_c = cache2[sly_ly-1][sly_sly-1][b]
    blk_o = cache_o2[sly_ly-1][sly_sly-1][b]

    rho_c   = pyconvert(Float64, blk_c.density)
    ye_c    = pyconvert(Float64, blk_c.eYield)
    mod_c   = pyconvert(Bool,    blk_c.isModified)
    slmod_c = pyconvert(Bool,    blk_c.isSubLayerModified)
    rho_o   = pyconvert(Float64, blk_o.density)
    ye_o    = pyconvert(Float64, blk_o.eYield)

    @assert abs(rho_c - new_rho_6) < 1e-12  "FAIL: sublayer block $(b+1) density not updated"
    @assert abs(ye_c  - new_ye_6)  < 1e-12  "FAIL: sublayer block $(b+1) eYield not updated"
    @assert mod_c   "FAIL: block $(b+1) isModified should be true"
    @assert slmod_c "FAIL: block $(b+1) isSubLayerModified should be true"
    @assert abs(rho_o - orig_rho_6) < 1e-4  "FAIL: original density was overwritten for block $(b+1)"
    @assert abs(ye_o  - orig_ye_6)  < 1e-4  "FAIL: original eYield was overwritten for block $(b+1)"
end
println("  ✓ All $n_blocks blocks updated in cache; original untouched.")

Sublayer [6,1] update applied.
─── Sublayer [6,1]: 18 blocks ────────────────────────────────
  ✓ All 18 blocks updated in cache; original untouched.


In [20]:
# ── Confirm: prior per-block edits in cache survive the sublayer update ───────
cache3     = earth.GetBlockMaterialCache()
cache_o3   = earth.GetBlockMaterialCacheOriginal()

println("─── Re-checking the 5 individually updated blocks after sublayer update ──")
for (idx, (i_ly, i_sly, i_blk, orig_rho, orig_ye)) in enumerate(targets)
    blk_c = cache3[i_ly-1][i_sly-1][i_blk-1]
    blk_o = cache_o3[i_ly-1][i_sly-1][i_blk-1]

    rho_c = pyconvert(Float64, blk_c.density)
    ye_c  = pyconvert(Float64, blk_c.eYield)
    rho_o = pyconvert(Float64, blk_o.density)
    ye_o  = pyconvert(Float64, blk_o.eYield)

    @assert abs(rho_c - new_densities[idx]) < 1e-12  "FAIL: block [$i_ly,$i_sly,$i_blk] cache density was reset by sublayer update"
    @assert abs(ye_c  - new_eyields[idx])   < 1e-12  "FAIL: block [$i_ly,$i_sly,$i_blk] cache eYield was reset by sublayer update"
    @assert abs(rho_o - orig_rho) < 1e-4             "FAIL: block [$i_ly,$i_sly,$i_blk] original density changed"
    @assert abs(ye_o  - orig_ye)  < 1e-4             "FAIL: block [$i_ly,$i_sly,$i_blk] original eYield changed"

    println("  Block [$i_ly,$i_sly,$i_blk]: cache ρ=$rho_c Ye=$ye_c  |  original ρ=$rho_o Ye=$ye_o  ✓")
end
println("\n✓ All previous per-block edits preserved. CacheOriginal unchanged throughout.")

─── Re-checking the 5 individually updated blocks after sublayer update ──
  Block [1,1,1]: cache ρ=111.0 Ye=0.11  |  original ρ=12.9807 Ye=0.4691  ✓
  Block [2,1,1]: cache ρ=222.0 Ye=0.22  |  original ρ=10.0861 Ye=0.4691  ✓
  Block [3,1,1]: cache ρ=333.0 Ye=0.33  |  original ρ=4.4872 Ye=0.4954  ✓
  Block [4,1,1]: cache ρ=444.0 Ye=0.44  |  original ρ=3.8983 Ye=0.4954  ✓
  Block [5,1,1]: cache ρ=555.0 Ye=0.55  |  original ρ=2.9 Ye=0.4956  ✓

✓ All previous per-block edits preserved. CacheOriginal unchanged throughout.


In [21]:
test = [
    (6, 1, 1, orig_rho_6, orig_ye_6),   # (i_ly, i_sly, i_blk, orig_rho, orig_ye)
    (6, 1, 2, orig_rho_6, orig_ye_6),
    (6, 1, 3,  orig_rho_6, orig_ye_6),
    (6, 1, 4,  orig_rho_6, orig_ye_6),
    (6, 1, 5,  orig_rho_6, orig_ye_6),
    (6, 1, 6, orig_rho_6, orig_ye_6),   # (i_ly, i_sly, i_blk, orig_rho, orig_ye)
    (6, 1, 7, orig_rho_6, orig_ye_6),
    (6, 1, 8,  orig_rho_6, orig_ye_6),
    (6, 1, 9,  orig_rho_6, orig_ye_6),
    (6, 1, 10,  orig_rho_6, orig_ye_6),
    (6, 1, 11, orig_rho_6, orig_ye_6),   # (i_ly, i_sly, i_blk, orig_rho, orig_ye)
    (6, 1, 12, orig_rho_6, orig_ye_6),
    (6, 1, 13,  orig_rho_6, orig_ye_6),
    (6, 1, 14,  orig_rho_6, orig_ye_6),
    (6, 1, 15,  orig_rho_6, orig_ye_6),
    (6, 1, 16,  orig_rho_6, orig_ye_6),
    (6, 1, 17,  orig_rho_6, orig_ye_6),
    (6, 1, 18,  orig_rho_6, orig_ye_6),
]


for (idx, (i_ly, i_sly, i_blk, orig_rho, orig_ye)) in enumerate(test)
    blk_c = cache3[i_ly-1][i_sly-1][i_blk-1]
    blk_o = cache_o3[i_ly-1][i_sly-1][i_blk-1]

    rho_c = pyconvert(Float64, blk_c.density)
    ye_c  = pyconvert(Float64, blk_c.eYield)
    rho_o = pyconvert(Float64, blk_o.density)
    ye_o  = pyconvert(Float64, blk_o.eYield)

    #@assert abs(rho_c - new_densities[idx]) < 1e-12  "FAIL: block [$i_ly,$i_sly,$i_blk] cache density was reset by sublayer update"
    #@assert abs(ye_c  - new_eyields[idx])   < 1e-12  "FAIL: block [$i_ly,$i_sly,$i_blk] cache eYield was reset by sublayer update"
    #@assert abs(rho_o - orig_rho) < 1e-4             "FAIL: block [$i_ly,$i_sly,$i_blk] original density changed"
    #@assert abs(ye_o  - orig_ye)  < 1e-4             "FAIL: block [$i_ly,$i_sly,$i_blk] original eYield changed"

    println("  Block [$i_ly,$i_sly,$i_blk]: cache ρ=$rho_c Ye=$ye_c  |  original ρ=$rho_o Ye=$ye_o  ✓")
end

  Block [6,1,1]: cache ρ=9.99 Ye=0.99  |  original ρ=1.02 Ye=0.5525  ✓
  Block [6,1,2]: cache ρ=9.99 Ye=0.99  |  original ρ=1.02 Ye=0.5525  ✓
  Block [6,1,3]: cache ρ=9.99 Ye=0.99  |  original ρ=1.02 Ye=0.5525  ✓
  Block [6,1,4]: cache ρ=9.99 Ye=0.99  |  original ρ=1.02 Ye=0.5525  ✓
  Block [6,1,5]: cache ρ=9.99 Ye=0.99  |  original ρ=1.02 Ye=0.5525  ✓
  Block [6,1,6]: cache ρ=9.99 Ye=0.99  |  original ρ=1.02 Ye=0.5525  ✓
  Block [6,1,7]: cache ρ=9.99 Ye=0.99  |  original ρ=1.02 Ye=0.5525  ✓
  Block [6,1,8]: cache ρ=9.99 Ye=0.99  |  original ρ=1.02 Ye=0.5525  ✓
  Block [6,1,9]: cache ρ=9.99 Ye=0.99  |  original ρ=1.02 Ye=0.5525  ✓
  Block [6,1,10]: cache ρ=9.99 Ye=0.99  |  original ρ=1.02 Ye=0.5525  ✓
  Block [6,1,11]: cache ρ=9.99 Ye=0.99  |  original ρ=1.02 Ye=0.5525  ✓
  Block [6,1,12]: cache ρ=9.99 Ye=0.99  |  original ρ=1.02 Ye=0.5525  ✓
  Block [6,1,13]: cache ρ=9.99 Ye=0.99  |  original ρ=1.02 Ye=0.5525  ✓
  Block [6,1,14]: cache ρ=9.99 Ye=0.99  |  original ρ=1.02 Ye=0.5525  ✓
 

### Batch vector setters — bounds-check test

`SetBlockDensity(vector<BlockMatUpdate>)` and `SetBlockEYield(vector<BlockMatUpdate>)`
validate every `(i_ly, i_sly, i_blk)` triple before applying the update.
Out-of-range entries are skipped with a `stderr` warning; valid entries still apply.

| Scenario | Expected behaviour |
|----------|--------------------||
| All indices valid | Every listed block gets the new value and `isModified = true` |
| One index out of range mixed with valid ones | Bad entry skipped; valid entries still applied |
| All indices out of range | Cache unchanged; warnings printed for each bad entry |

In [22]:
# ── Batch SetBlockDensity / SetBlockEYield — bounds-check test ───────────────
# Reset the cache so previous mutations do not interfere.
earth.ResetBlockMaterialCache()


test = [
    (6, 1, 1, orig_rho_6, orig_ye_6),   # (i_ly, i_sly, i_blk, orig_rho, orig_ye)
    (6, 1, 2, orig_rho_6, orig_ye_6),
    (6, 1, 3,  orig_rho_6, orig_ye_6),
    (6, 1, 4,  orig_rho_6, orig_ye_6),
    (6, 1, 5,  orig_rho_6, orig_ye_6),
    (6, 1, 6, orig_rho_6, orig_ye_6),   # (i_ly, i_sly, i_blk, orig_rho, orig_ye)
    (6, 1, 7, orig_rho_6, orig_ye_6),
    (6, 1, 8,  orig_rho_6, orig_ye_6),
    (6, 1, 9,  orig_rho_6, orig_ye_6),
    (6, 1, 10,  orig_rho_6, orig_ye_6),
    (6, 1, 11, orig_rho_6, orig_ye_6),   # (i_ly, i_sly, i_blk, orig_rho, orig_ye)
    (6, 1, 12, orig_rho_6, orig_ye_6),
    (6, 1, 13,  orig_rho_6, orig_ye_6),
    (6, 1, 14,  orig_rho_6, orig_ye_6),
    (6, 1, 15,  orig_rho_6, orig_ye_6),
    (6, 1, 16,  orig_rho_6, orig_ye_6),
    (6, 1, 17,  orig_rho_6, orig_ye_6),
    (6, 1, 18,  orig_rho_6, orig_ye_6),
]


for (idx, (i_ly, i_sly, i_blk, orig_rho, orig_ye)) in enumerate(test)
    blk_c = cache3[i_ly-1][i_sly-1][i_blk-1]
    blk_o = cache_o3[i_ly-1][i_sly-1][i_blk-1]

    rho_c = pyconvert(Float64, blk_c.density)
    ye_c  = pyconvert(Float64, blk_c.eYield)
    rho_o = pyconvert(Float64, blk_o.density)
    ye_o  = pyconvert(Float64, blk_o.eYield)

    #@assert abs(rho_c - new_densities[idx]) < 1e-12  "FAIL: block [$i_ly,$i_sly,$i_blk] cache density was reset by sublayer update"
    #@assert abs(ye_c  - new_eyields[idx])   < 1e-12  "FAIL: block [$i_ly,$i_sly,$i_blk] cache eYield was reset by sublayer update"
    #@assert abs(rho_o - orig_rho) < 1e-4             "FAIL: block [$i_ly,$i_sly,$i_blk] original density changed"
    #@assert abs(ye_o  - orig_ye)  < 1e-4             "FAIL: block [$i_ly,$i_sly,$i_blk] original eYield changed"

    println("  Block [$i_ly,$i_sly,$i_blk]: cache ρ=$rho_c Ye=$ye_c  |  original ρ=$rho_o Ye=$ye_o  ✓")
end


  Block [6,1,1]: cache ρ=1.02 Ye=0.5525  |  original ρ=1.02 Ye=0.5525  ✓
  Block [6,1,2]: cache ρ=1.02 Ye=0.5525  |  original ρ=1.02 Ye=0.5525  ✓
  Block [6,1,3]: cache ρ=1.02 Ye=0.5525  |  original ρ=1.02 Ye=0.5525  ✓
  Block [6,1,4]: cache ρ=1.02 Ye=0.5525  |  original ρ=1.02 Ye=0.5525  ✓
  Block [6,1,5]: cache ρ=1.02 Ye=0.5525  |  original ρ=1.02 Ye=0.5525  ✓
  Block [6,1,6]: cache ρ=1.02 Ye=0.5525  |  original ρ=1.02 Ye=0.5525  ✓
  Block [6,1,7]: cache ρ=1.02 Ye=0.5525  |  original ρ=1.02 Ye=0.5525  ✓
  Block [6,1,8]: cache ρ=1.02 Ye=0.5525  |  original ρ=1.02 Ye=0.5525  ✓
  Block [6,1,9]: cache ρ=1.02 Ye=0.5525  |  original ρ=1.02 Ye=0.5525  ✓
  Block [6,1,10]: cache ρ=1.02 Ye=0.5525  |  original ρ=1.02 Ye=0.5525  ✓
  Block [6,1,11]: cache ρ=1.02 Ye=0.5525  |  original ρ=1.02 Ye=0.5525  ✓
  Block [6,1,12]: cache ρ=1.02 Ye=0.5525  |  original ρ=1.02 Ye=0.5525  ✓
  Block [6,1,13]: cache ρ=1.02 Ye=0.5525  |  original ρ=1.02 Ye=0.5525  ✓
  Block [6,1,14]: cache ρ=1.02 Ye=0.5525  |  or

4. What the output of your hypothetical path function must look like
To be fully compatible with Pνν in flexOPT, the function must return:


paths::Vector{Path}    # length = length(cosθgrid)
zoa_reference::Float64 # the Z/A reference used to encode density (default 0.5, or 1.0 if storing nₑ directly)
Each Path encodes one trajectory as parallel segment vectors:


paths[l] = Path(
    density  :: Vector{Float64},   # one value per segment
    baseline :: Vector{Float64},   # one value per segment
)

In [23]:
# Build a vector of updates
updates_rho = ROOT.std.vector["ModularEarth::BlockMatUpdate"]()
updates_ye  = ROOT.std.vector["ModularEarth::BlockMatUpdate"]()

for (i_ly, i_sly, i_blk, rho, ye) in [
    (1, 1,  1, 13.0, 0.47),   # Inner Core, block 1
    (2, 1,  3, 10.5, 0.46),   # Outer Core, block 3
    (5, 1, 12,  3.1, 0.50),   # Lower Mantle, block 12
]
    u = ME.BlockMatUpdate()
    u.i_ly = i_ly;  u.i_sly = i_sly;  u.i_blk = i_blk

    u.value = rho;  updates_rho.push_back(u)
    u.value = ye;   updates_ye.push_back(u)
end

earth.SetBlockDensity(updates_rho)
earth.SetBlockEYield(updates_ye)

for (i_ly, i_sly, i_blk, rho, ye) in [
    (1, 1,  1, 13.0, 0.47),
    (2, 1,  3, 10.5, 0.46),
    (5, 1, 12,  3.1, 0.50),
]
    info = earth.GetBlockMatInfo(i_ly, i_sly, i_blk)
    println("[$i_ly,$i_sly,$i_blk]  density=$(pyconvert(Float64, info.density))  eYield=$(pyconvert(Float64, info.eYield))  isModified=$(pyconvert(Bool, info.isModified))")
    println("[$i_ly,$i_sly,$i_blk]  density=$(pyconvert(Float64, info.originalDensity))  eYield=$(pyconvert(Float64, info.originalEYield))  isModified=$(pyconvert(Bool, info.isModified))")

end


[1,1,1]  density=13.0  eYield=0.47  isModified=true
[1,1,1]  density=12.9807  eYield=0.4691  isModified=true
[2,1,3]  density=10.5  eYield=0.46  isModified=true
[2,1,3]  density=10.0861  eYield=0.4691  isModified=true
[5,1,12]  density=3.1  eYield=0.5  isModified=true
[5,1,12]  density=2.9  eYield=0.4956  isModified=true


### Test 4 — `GetSegments` with `radians = true`

The same direction expressed in radians (θ = π, φ = 0) must produce
an identical segment count and total path length as Test 3.
This confirms the degree/radian conversion branch.

In [24]:
# ── EarthPaths Test 4: GetSegments with radians=true ─────────────────────────


th_rad_val  = pi          # 180° in radians
phi_rad_val = 0.0

paths.GetSegments(th_rad_val, phi_rad_val, true)  # radians=true

segs_rad = paths.GetPathSegments()
N_rad    = pyconvert(Int, segs_rad.size())

total_rad = sum(pyconvert(Float64, segs_rad[i].distance) for i in 0:(N_rad-1))

println("Segments (radians call) : ", N_rad)
println("Total length (radians)  : ", round(total_rad; digits=3), " km")
println("Segments (degrees call) : ", N)    # from Test 3
println("Total length (degrees)  : ", round(total_dist; digits=3), " km")

@assert N_rad == N "FAIL: radians call gave different segment count ($N_rad vs $N)"
@assert abs(total_rad - total_dist) < 1e-6 "FAIL: radians/degrees path lengths differ ($total_rad vs $total_dist)"

println("\nTest 4 passed — radians=true gives identical result to degrees.")

Segments (radians call) : 12
Total length (radians)  : 12757.0 km
Segments (degrees call) : 12


LoadError: UndefVarError: `total_dist` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

### Test 5 — Oblique path (θ = 120°)

An oblique up-going neutrino (θ = 120°, φ = 0°) travels a chord shorter
than the full diameter.  Expected total length is in the range
[6371, 12757] km.

In [25]:
# ── EarthPaths Test 5: Oblique path θ=120° ───────────────────────────────────
paths.GetSegments(120.0, 0.0)   # degrees

segs_obl = paths.GetPathSegments()
N_obl    = pyconvert(Int, segs_obl.size())
total_obl = sum(pyconvert(Float64, segs_obl[i].distance) for i in 0:(N_obl-1))

println("Segments (θ=120°) : ", N_obl)
println("Total length      : ", round(total_obl; digits=3), " km")

@assert N_obl >= 1 "FAIL: no segments for oblique path"
@assert total_obl > 6371.0 "FAIL: oblique path shorter than Earth radius ($total_obl)"
@assert total_obl < 12757.0 + 500.0 "FAIL: oblique path longer than full diameter + margin ($total_obl)"

println("\nTest 5 passed — oblique path has ", N_obl, " segments, length ", round(total_obl; digits=1), " km.")

Segments (θ=120°) : 8
Total length      : 6400.895 km

Test 5 passed — oblique path has 8 segments, length 6400.9 km.


### Test 6 — `PrintSegments`

`PrintSegments()` writes a formatted table with columns
`Step | Distance[km] | Block | Sublayer | Layer | Material | Density[g/cm³]`
and a summary footer.  We re-run with the full-diameter path (θ = 180°)
so the output is maximal.

In [26]:
# ── EarthPaths Test 6: PrintSegments ─────────────────────────────────────────
# Re-run the full-diameter path so the table is interesting
paths.GetSegments(180.0, 0.0)

println("─── PrintSegments output ────────────────────────────────────────────────")
paths.PrintSegments()
println("─── end of table ────────────────────────────────────────────────────────")

# Verify it did not crash and segments are still accessible
n_after_print = pyconvert(Int, paths.GetPathSegments().size())
@assert n_after_print >= 2 "FAIL: PrintSegments seems to have emptied the segment list"

println("\nTest 6 passed — PrintSegments completed, ", n_after_print, " segments still in list.")

# ── New field checks: cumulative, Z, A ───────────────────────────────────────
segs6    = paths.GetPathSegments()
N6       = pyconvert(Int, segs6.size())

prev_sum = 0.0
for i in 0:(N6-1)
    s    = segs6[i]
    dist = pyconvert(Float64, s.distance)
    sum  = pyconvert(Float64, s.cumulative)
    Z    = pyconvert(Float64, s.Z)
    A    = pyconvert(Float64, s.A)

    # cumulative must grow strictly with each segment
    @assert sum > prev_sum "FAIL: cumulative not increasing at step $(i+1) (sum=$sum, prev=$prev_sum)"

    # cumulative must equal previous cumulative + this segment's distance
    @assert abs(sum - (prev_sum + dist)) < 1e-9 "FAIL: cumulative inconsistent at step $(i+1) (sum=$sum, prev+dist=$(prev_sum+dist))"

    # material must have positive Z and A
    @assert Z > 0.0 "FAIL: Z <= 0 at step $(i+1) (Z=$Z)"
    @assert A > 0.0 "FAIL: A <= 0 at step $(i+1) (A=$A)"

    prev_sum = sum
end

# last cumulative should equal the total path length reported by PrintSegments
last_sum = pyconvert(Float64, segs6[N6-1].cumulative)
println("\nLast segment cumulative : ", round(last_sum; digits=4), " km  (= total path length)")

# spot-check first segment's new fields
s0  = segs6[0]
println("First segment  Z=", pyconvert(Float64, s0.Z),
        "  A=", pyconvert(Float64, s0.A),
        "  cumulative=", pyconvert(Float64, s0.cumulative), " km")

println("\nTest 6 (new fields) passed — cumulative monotone and self-consistent; Z and A > 0 for all ", N6, " segments.")

─── PrintSegments output ────────────────────────────────────────────────
─── end of table ────────────────────────────────────────────────────────

Test 6 passed — PrintSegments completed, 12 segments still in list.

  Step  Distance[km]  Cumulative[km]  Block Sublayer  Layer                  MaterialDensity[g/cm3]       Z  A[g/mol]
---------------------------------------------------------------------------------------------------------------------
     1       15.0000         15.0000     15        1      7           BlockMat_7_1_15        0.0010  1.0000    0.0000
     2       15.0000         30.0000     15        1      6           BlockMat_6_1_15        1.0200  1.0000    0.0000
     3      385.0000        415.0000     15        1      5           BlockMat_5_1_15        2.9000  1.0000    0.0000
     4      270.0000        685.0000     15        1      4           BlockMat_4_1_15        3.8983  1.0000    0.0000
     5     2221.0000       2906.0000     15        1      3           Bloc

LoadError: AssertionError: FAIL: A <= 0 at step 1 (A=0.0)

### Test 7 — `GetPathSegments` field content

Access the `std::vector<PathSegment>` directly from Julia and verify the
named fields of the first and last segment.

| Field | Type | Meaning |
|-------|------|---------|
| `step` | `Int` | 1-based step counter |
| `distance` | `Float64` | Segment length [km] |
| `i_blk` | `Int` | Block index in sublayer (1-based) |
| `i_sly` | `Int` | Sublayer index in layer (1-based) |
| `i_ly` | `Int` | Layer index in Earth model (1-based) |
| `volName` | `String` | ROOT volume name `BlockVol_<ly>_<sly>_<blk>` |
| `matName` | `String` | Material name |
| `density` | `Float64` | Material density [g/cm³] |

In [27]:
# ── EarthPaths Test 7: PathSegment field access ──────────────────────────────
segs_final = paths.GetPathSegments()
N_final    = pyconvert(Int, segs_final.size())

@assert N_final > 0 "FAIL: No path segments returned by GetPathSegments()"

function inspect_segment(s, label)
    step     = pyconvert(Int, s.step)
    dist     = pyconvert(Float64, s.distance)
    i_blk    = pyconvert(Int, s.i_blk)
    i_sly    = pyconvert(Int, s.i_sly)
    i_ly     = pyconvert(Int, s.i_ly)

    vol_name = pyconvert(String, pybuiltins.str(s.volName))
    mat_name = pyconvert(String, pybuiltins.str(s.matName))

    density  = pyconvert(Float64, s.density)

    println("  $label")
    println("    step=$step  dist=$(round(dist; digits=4)) km")
    println("    layer=$i_ly  sublayer=$i_sly  block=$i_blk")
    println("    volume=\"$vol_name\"  material=\"$mat_name\"  ρ=$density g/cm³")

    # Verify that the volume name encodes the layer, sublayer, and block indices.
    expected_vol = "BlockVol_$(i_ly)_$(i_sly)_$(i_blk)"

    @assert vol_name == expected_vol "FAIL: volName ($vol_name) does not match indices (expected $expected_vol)"

    return nothing
end

println("Inspecting first and last segment of $N_final-segment path:")

inspect_segment(segs_final[0],         "First segment")
inspect_segment(segs_final[N_final-1], "Last segment")

println("\nTest 7 passed — all inspected PathSegment fields are accessible, and volName encodes the indices.")

Inspecting first and last segment of 12-segment path:
  First segment
    step=1  dist=15.0 km
    layer=7  sublayer=1  block=15
    volume="BlockVol_7_1_15"  material="BlockMat_7_1_15"  ρ=0.001 g/cm³
  Last segment
    step=12  dist=15.0 km
    layer=6  sublayer=1  block=1
    volume="BlockVol_6_1_1"  material="BlockMat_6_1_1"  ρ=1.02 g/cm³

Test 7 passed — all inspected PathSegment fields are accessible, and volName encodes the indices.


### Test 8 — `DisplayVectors`

`DisplayVectors()` creates a `TCanvas` and draws:
- The full Earth geometry (all layers via `Draw("ogl")`)
- A transparent atmosphere sphere shell at R_ATM = 6386 km
- The neutrino path line from `r_src` (cyan) → `r_det` (magenta) in blue
- A red propagation direction arrow (500 km long) at the source

In a headless / batch-mode environment the canvas is created in memory and
nothing is shown on screen.  The test verifies the call completes without
an exception and prints its diagnostics.

### EarthPaths test summary

| # | Capability tested | Assertion |
|---|-------------------|-----------|
| 1 | Constructor | Empty segment list before `GetSegments` |
| 2 | `SetDetectorLocation` | No exception; local frame built |
| 3 | `GetSegments` (degrees) | N ≥ 14; all fields valid; total length within 5 % |
| 4 | `GetSegments` (radians) | Identical result to the degrees call |
| 5 | Oblique path θ = 120° | Segments exist; length in [R⊕, 2R_ATM] |
| 6 | `PrintSegments` | Runs without error; list unchanged afterwards |
| 7 | `GetPathSegments` fields | `volName` encodes `i_ly`, `i_sly`, `i_blk` |
| 8 | `DisplayVectors` | TCanvas `'NuPath'` created in batch mode |

---
## `UpdateEarthPath` / `OriginalEarthPath` — Cache-based path re-evaluation

These two functions re-apply a material cache to the already-computed ROOT geometry,
so the expensive tracking step runs only once (inside `GetEarthPath`) and subsequent
material edits or baseline queries cost only a cache lookup.

| Function | Cache used | Use case |
|----------|-----------|----------|
| `UpdateEarthPath()` | `BlockMaterialCache` (live, editable) | Reflect any `SetBlock*` / `SetSublayer*` edits without re-tracking |
| `OriginalEarthPath()` | `BlockMaterialCacheOriginal` (frozen at `BuiltEarth()`) | Always returns the unmodified PREM propagation path |

**Call order requirement:** both functions require `GetEarthPath` to have been called first
to populate `MultiGeomPathCache`.  Calling them beforehand triggers a guard that returns
an empty vector and prints a diagnostic.

### Test 1 — Guard: calling before `GetEarthPath`

`MultiGeomPathCache` is empty here because `GetEarthPath` has not been called yet
in this section.  Both functions must return an empty `vector<PropaPath>` and print
a diagnostic to `stderr`.

In [28]:
# ── Test 1: Guard — before GetEarthPath ──────────────────────────────────────
# If a previous cell already populated MultiGeomPathCache this guard may not fire;
# the important thing is that neither function crashes on an empty cache.

println("─── UpdateEarthPath guard ───────────────────────────────────────────────")
result_guard = earth.UpdateEarthPath()
n_guard = pyconvert(Int, result_guard.size())
println("UpdateEarthPath returned $n_guard path(s)  (expected 0 when cache is empty)")

println("\n─── OriginalEarthPath guard ─────────────────────────────────────────────")
result_guard2 = earth.OriginalEarthPath()
n_guard2 = pyconvert(Int, result_guard2.size())
println("OriginalEarthPath returned $n_guard2 path(s)  (expected 0 when cache is empty)")

println("\n✓ Test 1 — guard check complete (see stderr for diagnostic messages).")


─── UpdateEarthPath guard ───────────────────────────────────────────────
UpdateEarthPath returned 0 path(s)  (expected 0 when cache is empty)

─── OriginalEarthPath guard ─────────────────────────────────────────────
OriginalEarthPath returned 0 path(s)  (expected 0 when cache is empty)

✓ Test 1 — guard check complete (see stderr for diagnostic messages).


[UpdateEarthPath] GeomCache is empty — call GetEarthPath first.
[OriginalEarthPath] GeomCache is empty — call GetEarthPath first.


### Test 2 — `GetEarthPath` setup + `OriginalEarthPath` returns PREM values

Call `GetEarthPath` with a 3-angle grid to populate `MultiGeomPathCache`, then
verify that `OriginalEarthPath` returns the unmodified PREM densities even before
any edits are applied.

PREM reference (prem_test table, Inner Core sublayer 1): ρ ≈ 12.9807 g/cm³, Ye ≈ 0.4691.

In [ ]:
# ── Test 2: GetEarthPath baseline + OriginalEarthPath ────────────────────────
# Reset any prior cache edits so we start from a clean PREM state.
earth.ResetBlockMaterialCache()

det_lon = 0.0    # longitude  [deg]
det_lat = 90.0   # latitude   [deg]  — North Pole
det_R   = 6371.0 # radius     [km]   — Earth's mean surface radius

earth.SetDetectorLocation(det_lon, det_lat, det_R)

# 3-angle grid: oblique (120°), steep (150°), vertical (180°).
th_vals = [180.0]

th_grid = ROOT.std.vector["double"]()
for th in th_vals;  th_grid.push_back(th);  end

phi = 0.0   # azimuth


0.0

In [30]:

# GetEarthPath runs ROOT tracking and stores geometry in MultiGeomPathCache.
paths_ref = earth.GetEarthPath(th_grid, phi)


[Earth::GetEarthPath] Building Earth paths for 1 angles.
[Earth::GetEarthPath] Free cache  paths for 1 angles.
Building PropaPaths for 0 angles.
Finished building PropaPath for 0 angles.
[Earth::GetEarthPath] Finished building paths for 1 angles.


[EarthPaths] GetSegments: detector location not set — call SetDetectorLocation() first.


Python: vector<ModularEarth::PropaPath>{ @0x4c7cdc50 }

In [ ]:


N_paths   = pyconvert(Int, paths_ref.size())

println("─── GetEarthPath ($(length(th_vals)) angles) ────────────────────────────────────────")
@assert N_paths == length(th_vals) "FAIL: expected $(length(th_vals)) PropaPath objects, got $N_paths"

for i in 0:(N_paths-1)
    pp     = paths_ref[i]
    N_segs = pyconvert(Int, pp.density.size())
    total  = sum(pyconvert(Float64, pp.baseline[j]) for j in 0:(N_segs-1))
    println("  Path $(i+1) θ=$(th_vals[i+1])°: $(N_segs) segments, total $(round(total; digits=1)) km")
    @assert N_segs >= 1 "FAIL: path $(i+1) has no segments"
end

# ── OriginalEarthPath ─────────────────────────────────────────────────────────
orig_paths = earth.OriginalEarthPath()
N_orig     = pyconvert(Int, orig_paths.size())

println("\n─── OriginalEarthPath ───────────────────────────────────────────────────")
@assert N_orig == N_paths "FAIL: OriginalEarthPath returned $N_orig paths, expected $N_paths"

# Spot-check: first segment of the vertical path (θ=180°) enters the inner core.
prem_rho_ic = 12.9807
prem_ye_ic  = 0.4691

pp_vert       = orig_paths[N_paths-1]          # 0-based; θ=180° is last
N_segs_vert   = pyconvert(Int, pp_vert.density.size())
rho0_orig     = pyconvert(Float64, pp_vert.density[0])
ye0_orig      = pyconvert(Float64, pp_vert.eyield[0])
ne0_orig      = pyconvert(Float64, pp_vert.ne[0])

println("  Vertical path (θ=180°): $N_segs_vert segments")
println("  Seg[0] density = $rho0_orig  (expected ≈ $prem_rho_ic)")
println("  Seg[0] eYield  = $(round(ye0_orig; digits=4))  (expected ≈ $prem_ye_ic)")
println("  Seg[0] ne      = $(round(ne0_orig; digits=4))  (expected ≈ $(round(prem_rho_ic*prem_ye_ic; digits=4)))")

@assert abs(rho0_orig - prem_rho_ic) < 1e-3  "FAIL: OriginalEarthPath density ($rho0_orig) ≠ PREM ($prem_rho_ic)"
@assert abs(ye0_orig  - prem_ye_ic)  < 1e-3  "FAIL: OriginalEarthPath eYield ($ye0_orig) ≠ PREM ($prem_ye_ic)"
@assert abs(ne0_orig  - prem_rho_ic * prem_ye_ic) < 1e-3  "FAIL: ne ($ne0_orig) ≠ ρ·Ye ($(prem_rho_ic*prem_ye_ic))"

println("\n✓ Test 2 passed — OriginalEarthPath returns PREM values for unmodified model.")


### Test 3 — `UpdateEarthPath` reflects material cache edits; `OriginalEarthPath` unchanged

After calling `SetBlockDensity` on a block that lies on the tracked path,
`UpdateEarthPath` must return the new density while `OriginalEarthPath`
continues to return the original PREM value.

In [ ]:
# ── Test 3: UpdateEarthPath reflects SetBlockDensity ─────────────────────────
# Modify inner core block [1,1,1] — always the first segment of the vertical path.
new_rho_ic = 99.0
new_ye_ic  = 0.30
earth.SetBlockDensity(1, 1, 1, new_rho_ic)
earth.SetBlockEYield(1, 1, 1, new_ye_ic)
println("─── Applied SetBlockDensity/EYield(1,1,1) → ρ=$new_rho_ic, Ye=$new_ye_ic ──────")

# UpdateEarthPath: must reflect the edit without re-running ROOT tracking.
upd_paths = earth.UpdateEarthPath()
N_upd     = pyconvert(Int, upd_paths.size())
@assert N_upd == N_paths "FAIL: UpdateEarthPath returned $N_upd paths, expected $N_paths"

pp_upd     = upd_paths[N_paths-1]
rho0_upd   = pyconvert(Float64, pp_upd.density[0])
ye0_upd    = pyconvert(Float64, pp_upd.eyield[0])
ne0_upd    = pyconvert(Float64, pp_upd.ne[0])

println("  UpdateEarthPath  Seg[0]: ρ=$rho0_upd  Ye=$ye0_upd  ne=$(round(ne0_upd;digits=4))")
@assert abs(rho0_upd - new_rho_ic) < 1e-12  "FAIL: UpdateEarthPath density ($rho0_upd) ≠ $new_rho_ic"
@assert abs(ye0_upd  - new_ye_ic)  < 1e-12  "FAIL: UpdateEarthPath eYield ($ye0_upd) ≠ $new_ye_ic"
@assert abs(ne0_upd  - new_rho_ic * new_ye_ic) < 1e-12  "FAIL: ne ($ne0_upd) ≠ ρ·Ye ($(new_rho_ic*new_ye_ic))"

# OriginalEarthPath: must still return unmodified PREM — the edit must not bleed through.
orig_paths2  = earth.OriginalEarthPath()
pp_orig2     = orig_paths2[N_paths-1]
rho0_orig2   = pyconvert(Float64, pp_orig2.density[0])
ye0_orig2    = pyconvert(Float64, pp_orig2.eyield[0])

println("  OriginalEarthPath Seg[0]: ρ=$rho0_orig2  Ye=$(round(ye0_orig2;digits=4))  (expected ≈ $prem_rho_ic / $prem_ye_ic)")
@assert abs(rho0_orig2 - prem_rho_ic) < 1e-3  "FAIL: OriginalEarthPath density changed after edit ($rho0_orig2 ≠ $prem_rho_ic)"
@assert abs(ye0_orig2  - prem_ye_ic)  < 1e-3  "FAIL: OriginalEarthPath eYield changed after edit ($ye0_orig2 ≠ $prem_ye_ic)"

println("\n✓ Test 3 passed — UpdateEarthPath sees the edit; OriginalEarthPath is unaffected.")


### Test 4 — `costh` / `azimuth` correctness and cross-call isolation

Each `PropaPath` returned by both functions must carry `costh = cos(θ)` and
`azimuth = φ` matching the grid passed to `GetEarthPath`.
Additionally, calling `OriginalEarthPath` must not alter the material state
visible to a subsequent `UpdateEarthPath` call.

In [ ]:
# ── Test 4: costh / azimuth + cross-call isolation ───────────────────────────
println("─── costh / azimuth consistency ─────────────────────────────────────────")
for i in 0:(N_paths-1)
    th_deg         = th_vals[i+1]
    expected_costh = cos(deg2rad(th_deg))

    costh_upd  = pyconvert(Float64, upd_paths[i].costh)
    costh_orig = pyconvert(Float64, orig_paths2[i].costh)
    az_upd     = pyconvert(Float64, upd_paths[i].azimuth)
    az_orig    = pyconvert(Float64, orig_paths2[i].azimuth)

    println("  θ=$(th_deg)°  expected costh=$(round(expected_costh; digits=5))")
    println("    UpdateEarthPath:   costh=$(round(costh_upd; digits=5))  azimuth=$az_upd")
    println("    OriginalEarthPath: costh=$(round(costh_orig; digits=5))  azimuth=$az_orig")

    @assert abs(costh_upd  - expected_costh) < 1e-10  "FAIL: UpdateEarthPath costh wrong for θ=$th_deg°"
    @assert abs(costh_orig - expected_costh) < 1e-10  "FAIL: OriginalEarthPath costh wrong for θ=$th_deg°"
    @assert abs(az_upd  - phi) < 1e-12  "FAIL: UpdateEarthPath azimuth wrong (got $az_upd)"
    @assert abs(az_orig - phi) < 1e-12  "FAIL: OriginalEarthPath azimuth wrong (got $az_orig)"
end
println("  ✓ costh and azimuth correct for all $(N_paths) paths in both functions.")

# Cross-call isolation: calling OriginalEarthPath must not reset the live cache edit.
println("\n─── Cross-call isolation ────────────────────────────────────────────────")
_ = earth.OriginalEarthPath()   # extra OriginalEarthPath call
upd_check = earth.UpdateEarthPath()
rho_check = pyconvert(Float64, upd_check[N_paths-1].density[0])
@assert abs(rho_check - new_rho_ic) < 1e-12  "FAIL: UpdateEarthPath density changed after calling OriginalEarthPath ($rho_check ≠ $new_rho_ic)"
println("  ✓ UpdateEarthPath still sees ρ=$rho_check after an OriginalEarthPath call.")

# Clean up for any further cells.
earth.ResetBlockMaterialCache()
println("  Cache reset to PREM baseline.")

println("\n✓ Test 4 passed — costh/azimuth correct; functions are fully independent.")


### `UpdateEarthPath` / `OriginalEarthPath` test summary

| # | Capability tested | Assertion |
|---|-------------------|-----------|
| 1 | Guard before `GetEarthPath` | Both return empty vector; diagnostic printed |
| 2 | `OriginalEarthPath` on clean model | First segment ρ and Ye match PREM; ne = ρ·Ye |
| 3 | `UpdateEarthPath` reflects `SetBlockDensity/EYield` | Modified values appear; `OriginalEarthPath` unchanged |
| 4 | `costh` / `azimuth` correctness | Match `cos(θ)` and `φ` passed to `GetEarthPath` |
| 4 | Cross-call isolation | `OriginalEarthPath` does not reset the live cache visible to `UpdateEarthPath` |